# One assay, one honest QSAR pilot
**Human dopamine D2 · ChEMBL CHEMBL3855278 · experimental binding pKi, standardized by ChEMBL to Ki.**

This notebook fits a small fixed model, not a data-retrieval demo. The input is one frozen assay, not pooled targets or endpoints. Source-reported pKi, conversion checks, calculated descriptors and predictions remain separate. The old 250-compound model and public explorer are unchanged.

Run beside `pilot.py`, `input_hashes.json`, `predeclared_recipe.json` and `frozen/` from the supplied bundle. **Offline replay:** use the tested environment in `requirements-tested.txt`; no network, EXE, DLL or installation runs in these cells. Fresh Colab is a separate, untested environment: install the listed Python packages there and place the bundle in its working directory before Run all. Nothing is uploaded automatically.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
import pilot
ROOT = Path.cwd()
assert (ROOT / 'input_hashes.json').is_file(), 'Run from the extracted bundle directory.'
pilot.verify_frozen(ROOT)
recipe = json.loads((ROOT / 'predeclared_recipe.json').read_text())
print(recipe['models'], '\nNo tuning; preprocessing sees training data only.')


## 1. Keep the measurement trail
The frozen cohort ledger retains original and standardized values, relations, units, activity IDs and structure decisions. Source-reported pKi is experimental evidence; ChEMBL standardized Ki (nM) is a representation of that measurement, not a new experiment. Only exact source-reported pKi with a valid matching standardized Ki enters the fit; censored records do not become exact labels. For Ki in nM, **pKi = 9 − log10(Ki)**. The model uses original source-reported pKi, avoiding artificial precision from the back-conversion; round-trip pKi is a separate check that can differ slightly through rounding. Source pKi is not a model prediction. All full-isomeric parent identities with multiple source records are excluded regardless of label agreement; unresolved pairs are not averaged. Repeated records and exclusions remain inspectable in `frozen/`; see its cohort report for exact decisions and protocol-verification limits.


In [ ]:
cohort = pd.read_csv(ROOT / 'frozen' / 'eligible_compounds.csv')
display(cohort.head())
print(f'{len(cohort)} eligible compounds; one assay/document only.')


## 2. Freeze chemistry and split before fitting
Five explicit descriptors use Pat Walters’ `useful_rdkit_utils.RDKitDescriptors`: molecular weight, calculated LogP, TPSA, H-bond donors and acceptors. Descriptors use the explicitly curated parent SMILES; source structures and the parent-selection decision remain in the ledger. No extra neutralization or tautomerization is performed by the modeling step.

The predeclared algorithm groups nonchiral Murcko scaffolds **and** parent/connectivity identities, orders groups by a seeded SHA-256 hash, then allocates whole groups toward 20% holdout while retaining at least 10 training and 5 test compounds. Only if group sizes make that impossible, use parent/connectivity groups and disclose congeneric interpolation limits. No activity value chooses the split. Exact implementation is frozen in `pilot.py`.


In [ ]:
dataset, molecules, split_info = pilot.prepare(ROOT)
display(pd.Series(split_info, name='Pre-fit split'))
display(dataset.groupby('split')[pilot.FEATURES].agg(['count', 'mean']))


## 3. Fit once; compare against the training mean
`StandardScaler → Ridge(alpha=1)` is fixed in advance. No feature selection, hyperparameter search or holdout-driven changes. The baseline always predicts the **training** mean. Both models are scored on exactly the same untouched holdout. There is no final all-data refit.


In [ ]:
report = pilot.run(ROOT)
display(pd.DataFrame(report['holdout_metrics']).T.round(4))
print(f"Train: {report['train_n']}; test: {report['test_n']}; total: {report['cohort_n']}")


**Observed result:** 80 compounds → 57 train and 23 holdout. Whole-group allocation produced 28.75% holdout rather than exactly 20%; the holdout covers only **6 scaffolds**. Ridge MAE/RMSE = **0.370/0.491 pKi**, versus **0.376/0.497** for the training mean. R² is negative for both (−0.052 and −0.077). The small numerical advantage does **not** establish useful predictive skill. Predictions stay near the training mean and miss the observed potency range. No alternate split/model was tried to improve these results.


## 4. Inspect errors, not just a score
MAE/RMSE are in pKi units; lower is better. R² can be negative. A tiny single-assay holdout is not external validation and cannot establish superiority over the unrelated old model. Descriptor-range and nearest-training Morgan similarity are **descriptive warnings**, not calibrated uncertainty or proof of applicability.


In [ ]:
predictions = pd.read_csv(ROOT / 'results' / 'predictions.csv')
heldout = predictions.query("split == 'test'")
display(heldout[['molecule_chembl_id', 'pKi', 'predicted_pKi_ridge',
    'predicted_pKi_train_mean', 'residual_observed_minus_predicted',
    'outside_training_descriptor_range', 'max_train_morgan_similarity']])
display(Image(filename=str(ROOT / 'results' / 'holdout.png')))


## 5. Reuse the result carefully
`results/` contains the model dataset, split identities, predictions/residuals, metrics, plot and fitted model. `model_parameters.json` is the transparent scaler/coefficient export; `model.joblib` is a Python serialization—load only your own trusted copy. The hash gate verifies input bytes, not scientific truth.

The measured cohort, curation ledger and source provenance travel with the bundle. This is a reproducible exploratory single-assay pilot: its value is an honest baseline comparison, even when Ridge loses. New-assay validation, a larger independent test set and chemical-context review are needed before practical prediction claims.

**Attribution:** descriptor API from [Pat Walters, Useful RDKit Utils](https://github.com/PatWalters/useful_rdkit_utils/tree/1057ec09a6d55768ecbfa0bfe4f0d3ef3f20163f), reviewed version 2.0.1. Workflow code is newly written; no tutorial benchmark is claimed reproduced. Source data: ChEMBL, assay CHEMBL3855278, document CHEMBL3853329; provenance and licensing notes are retained in the frozen bundle.


In [ ]:
for limitation in report['limitations']:
    print('•', limitation)
print('Exports:', ', '.join(p.name for p in sorted((ROOT / 'results').iterdir())))


## Source-provenance check after the first fit

The first model was fitted after the complete cohort was handed off as ready. A later source check identified a possible literature comparator, named GSK598809A in the manuscript, so final release was paused. The registered GSK598809 identity (PubChem CID 11784937) was checked against all 107 source structures: no exact full-identity or stereo-ignored parent-graph match was found. Independent audit confirmed that absence. No exclusion, split change, parameter change or corrective refit was justified or performed.

The suffix “A” and source figure were not visually confirmed, and complete source-table-to-record mapping remains unresolved. This is a **database-attributed, single-assay experimental-label pilot**, not independently verified original laboratory measurements. The original 80-compound cohort and first-fit results remain unchanged. Machine-readable identity checks and source response bytes accompany this note.
